In [0]:
-- KPI: Profit Growth % MoM
-- Purpose:
-- Calculates month-over-month profit growth by comparing
-- the latest month with the previous month.

WITH monthly_profit AS (
    SELECT
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(profit) AS total_profit
    FROM `end-to-end_pipeline`.gold.fact_sales
    GROUP BY DATE_TRUNC('month', order_date)
),

profit_comparison AS (
    SELECT
        sales_month,
        total_profit,
        LAG(total_profit) OVER (
            ORDER BY sales_month
        ) AS previous_month_profit
    FROM monthly_profit
)

SELECT
    ROUND(
        ((total_profit - previous_month_profit)
        / NULLIF(previous_month_profit, 0)) * 100,
        2
    ) AS profit_growth_mom_pct
FROM profit_comparison
WHERE previous_month_profit IS NOT NULL
ORDER BY sales_month DESC
LIMIT 1;